# Reliance Industries Stock Price Prediction
### Comparing MLP, LSTM and GRU

**Goal:** predict the next trading day's closing price from the previous 60 closing prices, and compare three neural network architectures against a simple persistence baseline.

**Pipeline**
1. Load and clean data
2. Exploratory analysis
3. Scaling and chronological train/test split
4. Sliding-window sequences
5. Build MLP, LSTM, GRU
6. Train with early stopping
7. Evaluate and compare (MAE, RMSE, MAPE, direction accuracy)
8. Save models for the Streamlit app
9. Predict the next day

## 1. Imports and configuration

In [ ]:
import os
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Flatten, LSTM, GRU
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
WINDOW_SIZE = 60
TRAIN_RATIO = 0.80
EPOCHS = 100
BATCH_SIZE = 32

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.makedirs("models", exist_ok=True)
os.makedirs("plots", exist_ok=True)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

## 2. Load and clean the dataset

In [ ]:
df = pd.read_csv("stock_data.csv")
df.columns = [str(c).strip() for c in df.columns]

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
print("Data types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Dates in this file are dd-mm-yyyy, so dayfirst=True is required
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

df = df.dropna(subset=["Date", "Close"])
df = df[df["Close"] > 0]
df = df.drop_duplicates(subset=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print("Clean shape:", df.shape)
print("Start date:", df["Date"].min().date())
print("End date:  ", df["Date"].max().date())
df[["Date", "Close"]].describe()

## 3. Exploratory analysis

In [ ]:
fig, ax = plt.subplots()
ax.plot(df["Date"], df["Close"], label="Close", linewidth=1)
ax.plot(df["Date"], df["Close"].rolling(50).mean(), label="MA50", linewidth=1)
ax.plot(df["Date"], df["Close"].rolling(200).mean(), label="MA200", linewidth=1)
ax.set_title("Reliance closing price")
ax.set_xlabel("Date")
ax.set_ylabel("Price (Rs.)")
ax.legend()
plt.tight_layout()
plt.savefig("plots/price_history.png")
plt.show()

In [ ]:
returns = df["Close"].pct_change().dropna() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(returns, bins=100)
axes[0].set_title("Daily returns distribution (%)")
axes[1].plot(df["Date"].iloc[1:], returns, linewidth=0.5)
axes[1].set_title("Daily returns over time (%)")
plt.tight_layout()
plt.show()

print(f"Mean daily return: {returns.mean():.3f}%")
print(f"Std of daily return: {returns.std():.3f}%")

## 4. Scaling and chronological split

The data is split **in time order** (first 80% train, last 20% test) with no shuffling, so the model never sees the future. The scaler is fitted **only on the training prices** to avoid data leakage.

In [ ]:
prices = df["Close"].to_numpy().reshape(-1, 1)
split_index = int(len(prices) * TRAIN_RATIO)

scaler = MinMaxScaler()
scaler.fit(prices[:split_index])
scaled_prices = scaler.transform(prices)

print("Train days:", split_index)
print("Test days: ", len(prices) - split_index)
print("Train ends:", df["Date"].iloc[split_index - 1].date())
print("Test starts:", df["Date"].iloc[split_index].date())

## 5. Create sequences
Each sample is 60 consecutive scaled prices; the target is the next day's scaled price.

In [ ]:
X_train, y_train, X_test, y_test = [], [], [], []

for i in range(WINDOW_SIZE, len(scaled_prices)):
    seq = scaled_prices[i - WINDOW_SIZE:i]
    target = scaled_prices[i]
    if i < split_index:
        X_train.append(seq)
        y_train.append(target)
    else:
        X_test.append(seq)
        y_test.append(target)

X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.float32)
X_test = np.array(X_test, dtype=np.float32)
y_test = np.array(y_test, dtype=np.float32)

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test: ", X_test.shape, " y_test: ", y_test.shape)

## 6. Define the models

In [ ]:
def build_mlp():
    return Sequential([
        Input(shape=(WINDOW_SIZE, 1)),
        Flatten(),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1),
    ])


def build_lstm():
    return Sequential([
        Input(shape=(WINDOW_SIZE, 1)),
        LSTM(50, return_sequences=True),
        LSTM(50),
        Dense(1),
    ])


def build_gru():
    return Sequential([
        Input(shape=(WINDOW_SIZE, 1)),
        GRU(50, return_sequences=True),
        GRU(50),
        Dense(1),
    ])


models = {"MLP": build_mlp(), "LSTM": build_lstm(), "GRU": build_gru()}

for name, m in models.items():
    print(f"{name}: {m.count_params():,} parameters")

## 7. Train all three models

In [ ]:
histories = {}

for name, model in models.items():
    print("\n" + "=" * 50)
    print(f"TRAINING {name}")
    print("=" * 50)

    model.compile(optimizer="adam", loss="mean_squared_error")

    early_stopping = EarlyStopping(monitor="val_loss", patience=10,
                                   restore_best_weights=True)

    histories[name] = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        shuffle=False,
        callbacks=[early_stopping],
        verbose=1,
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (name, h) in zip(axes, histories.items()):
    ax.plot(h.history["loss"], label="Training loss")
    ax.plot(h.history["val_loss"], label="Validation loss")
    ax.set_title(f"{name} ({len(h.history['loss'])} epochs)")
    ax.set_xlabel("Epoch")
    ax.legend()
axes[0].set_ylabel("MSE")
plt.tight_layout()
plt.savefig("plots/training_loss.png")
plt.show()

## 8. Evaluate on the test set

In [ ]:
actual_prices = scaler.inverse_transform(y_test).flatten()
prev_prices = prices[split_index - 1:-1].flatten()   # yesterday's close for each test day

predictions = {}
for name, model in models.items():
    p = model.predict(X_test, verbose=0)
    predictions[name] = scaler.inverse_transform(p).flatten()

# Persistence baseline: tomorrow = today
predictions["Persistence Baseline"] = prev_prices


def evaluate(name, pred):
    mae = mean_absolute_error(actual_prices, pred)
    rmse = np.sqrt(mean_squared_error(actual_prices, pred))
    mape = np.mean(np.abs(actual_prices - pred) / actual_prices) * 100
    if name == "Persistence Baseline":
        direction = np.nan
    else:
        direction = np.mean(np.sign(pred - prev_prices) == np.sign(actual_prices - prev_prices)) * 100
    return {"Model": name, "MAE": mae, "RMSE": rmse,
            "MAPE (%)": mape, "Direction Accuracy (%)": direction}


results_df = pd.DataFrame([evaluate(n, p) for n, p in predictions.items()])
results_df = results_df.sort_values("MAE").reset_index(drop=True)
results_df.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ["#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]
for ax, metric in zip(axes, ["MAE", "RMSE"]):
    bars = ax.bar(results_df["Model"], results_df[metric], color=colors)
    ax.bar_label(bars, fmt="%.2f")
    ax.set_title(f"{metric} (lower is better)")
    ax.set_ylabel("Rs.")
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig("plots/metric_comparison.png")
plt.show()

In [ ]:
test_dates = df["Date"].iloc[split_index:].to_numpy()

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

axes[0].plot(test_dates, actual_prices, label="Actual", color="black", linewidth=1.5)
for name in ["MLP", "LSTM", "GRU"]:
    axes[0].plot(test_dates, predictions[name], label=name, linewidth=1)
axes[0].set_title("Actual vs predicted: full test period")
axes[0].legend()

N = 100
axes[1].plot(test_dates[-N:], actual_prices[-N:], label="Actual", color="black", linewidth=2)
for name in ["MLP", "LSTM", "GRU"]:
    axes[1].plot(test_dates[-N:], predictions[name][-N:], label=name, linewidth=1.2)
axes[1].set_title(f"Actual vs predicted: last {N} test days")
axes[1].legend()

for ax in axes:
    ax.set_ylabel("Price (Rs.)")
plt.tight_layout()
plt.savefig("plots/actual_vs_predicted.png")
plt.show()

In [ ]:
best = results_df.iloc[0]
best_nn = results_df[results_df["Model"] != "Persistence Baseline"].iloc[0]

print(f"Lowest MAE overall      : {best['Model']} ({best['MAE']:.2f})")
print(f"Best neural network     : {best_nn['Model']} (MAE {best_nn['MAE']:.2f})")

if best["Model"] == "Persistence Baseline":
    print("\nThe simple baseline beat every neural network on raw error.")
    print("This is common: stock prices behave close to a random walk.")
else:
    print(f"\n{best['Model']} beat the persistence baseline.")

## 9. Save models, scaler and results
These files are used by the Streamlit app (`app.py`).

In [ ]:
for name, model in models.items():
    model.save(f"models/{name.lower()}_model.keras")

joblib.dump(scaler, "models/price_scaler.pkl")
results_df.to_csv("model_results.csv", index=False)

print("Saved:", sorted(os.listdir("models")))

## 10. Predict the next trading day

In [ ]:
latest_prices = df["Close"].tail(WINDOW_SIZE).to_numpy().reshape(-1, 1)
x_input = scaler.transform(latest_prices).reshape(1, WINDOW_SIZE, 1)

last_date = df["Date"].iloc[-1]
last_close = float(latest_prices[-1, 0])

rows = []
for name, model in models.items():
    p = scaler.inverse_transform(model.predict(x_input, verbose=0).reshape(-1, 1))[0, 0]
    rows.append({"Model": name,
                 "Predicted next close": round(float(p), 2),
                 "Change": round(float(p - last_close), 2),
                 "Change (%)": round(float((p - last_close) / last_close * 100), 2)})

next_day = pd.DataFrame(rows)
print(f"Latest data date : {last_date.date()}")
print(f"Latest close     : Rs. {last_close:.2f}\n")
next_day

In [ ]:
next_day.to_csv("next_day_prediction.csv", index=False)
print("Saved next_day_prediction.csv")

## 11. Conclusions

- Compare the **MAE / RMSE / MAPE** table above to see which architecture fits best.
- **Direction accuracy** shows how often each model gets the up/down move right; around 50% means no better than a coin flip.
- The **persistence baseline** is a tough benchmark for price forecasting. If it wins, that is a finding, not a bug.
- Only closing prices were used. Possible improvements: add volume, returns or technical indicators, predict returns instead of prices, tune the window size, or use walk-forward validation.

> Educational project. Not financial advice.

**Next step:** run the dashboard with `streamlit run app.py`.